In [ ]:
import os
import torch
from datetime import datetime
from pathlib import Path

# Impor dari modul-modul yang sudah dibuat
from config import USE_SHP_FOR_USER, SHP_PATH, N_USER_POINTS, USER_JSON_PATH
from device_config import DEVICE, info_device
from model import PolicyNetwork
from training import meta_train
from evaluation import evaluate_only_and_log
from osrm_utils import load_osrm_cache, save_osrm_cache

# Impor fungsi-fungsi untuk memuat data (sesuai struktur Anda)
# Pastikan file `data_loader.py` ada di direktori yang sama.
from data_loader import (
    load_mmi_from_cont_mmi,
    load_evac_candidates_geojson,
    load_user_coords,
    generate_user_coords_from_shp,
    generate_and_save_user_coords_if_needed,
)

In [ ]:
info_device()

# --- Konfigurasi path dan daftar event ---
DATA_DIR = "./kota-surabaya"
SAVE_DIR = "./hasil_training"
EVENT_LIST = [
    "us6000k49j",
    "us6000mkfz",
    # "us60005kta", # Contoh event lain
    # "usc000nahz",
]

# Pastikan direktori penyimpanan ada
os.makedirs(SAVE_DIR, exist_ok=True)

# Muat OSRM cache untuk mempercepat kalkulasi jarak
# Sesuaikan path jika perlu
load_osrm_cache("./osrm_cache/osrm_cache_baru_shp.json")

In [ ]:
print("Mempersiapkan koordinat pengguna...")
generate_and_save_user_coords_if_needed(SHP_PATH, USER_JSON_PATH, n=N_USER_POINTS)


def build_task_list(event_list, data_dir, evac_geojson_path):
    """
    Membangun daftar tasks dari data event, dengan sumber data evakuasi
    dan MMI yang terpisah.
    """
    if USE_SHP_FOR_USER:
        print("📍 Menggunakan SHP untuk menghasilkan titik pengguna acak...")
        user_coords = generate_user_coords_from_shp(SHP_PATH, n=N_USER_POINTS)
    else:
        print("📍 Menggunakan JSON untuk titik pengguna tetap...")
        user_coords = load_user_coords(USER_JSON_PATH)

    # Muat kandidat evakuasi dari GeoJSON sekali saja (untuk semua event)
    evac_candidates = load_evac_candidates_geojson(evac_geojson_path)
    print(f"✅ Berhasil memuat {len(evac_candidates)} kandidat evakuasi.")

    tasks = []
    for event_id in event_list:
        try:
            # Muat data MMI spesifik untuk setiap event
            mmi_json_path = os.path.join(data_dir, f"{event_id}-cont_mmi.json")
            mmi_points = load_mmi_from_cont_mmi(mmi_json_path)

            tasks.append(
                {
                    "event_id": event_id,
                    "user_coords": user_coords,
                    "evac_candidates": evac_candidates,
                    "mmi_points": mmi_points,  # Data mentah MMI
                }
            )
            print(f"✅ Task untuk event '{event_id}' berhasil dibuat.")
        except Exception as e:
            print(f"⚠️ [SKIP] Event {event_id} gagal dimuat: {e}")

    return tasks


# --- Bangun daftar task ---
print("\nMembangun daftar tasks untuk meta-training...")
TASK_LIST = build_task_list(
    EVENT_LIST,
    DATA_DIR,
    evac_geojson_path=os.path.join(DATA_DIR, "DATA-WONOCOLO.geojson"),
)

In [ ]:
if TASK_LIST:
    print("\n🚀 Memulai proses Meta-Training...")
    meta_model = meta_train(
        TASK_LIST, save_path=os.path.join(SAVE_DIR, "meta_model_final.pt")
    )

    # Simpan model dengan nama yang menyertakan tanggal
    today_str = datetime.today().strftime("%Y-%m-%d")
    final_save_path = Path(SAVE_DIR) / f"meta_model_{today_str}.pt"
    torch.save(meta_model.state_dict(), final_save_path)
    print(f"\n💾 Model Meta-RL terakhir disimpan di: {final_save_path}")
else:
    print("\n⚠️ Tidak ada task yang valid untuk training. Proses dihentikan.")
    meta_model = None

In [ ]:
if meta_model:
    print("\n🔍 Memulai evaluasi model pada setiap task...")
    for task in TASK_LIST:
        print("-" * 50)
        evaluate_only_and_log(
            model=meta_model,
            user_coords=task["user_coords"],
            evac_candidates=task["evac_candidates"],
            mmi_points=task["mmi_points"],
            event_id=task["event_id"],
            output_dir="evaluation_logs",
        )
else:
    print("\n⚠️ Model tidak dilatih, proses evaluasi dilewati.")

In [ ]:
print("\nMenyimpan cache OSRM yang diperbarui...")
save_osrm_cache("./osrm_cache/osrm_cache_baru_shp.json")
print("\n🎉 Semua proses telah selesai!")